# DriveSense-VLM — 00: Data Pipeline

**GPU**: T4 (or CPU) | **Time**: ~2–3 h | **Cost**: ~5 CU

> ⚠️ **Use a FRESH runtime.** Never downgrade numpy — Colab ships numpy 2.x and all
> packages are compiled against it. Set `FORCE_RERUN = True` to rebuild any step.

Pipeline:
1. nuScenes mini download + rarity filtering
2. PySpark distributed ETL + analytics
3. DADA-2000 critical-moment extraction (optional, 53 GB)
4. Unified dataset assembly (stratified train/val/test splits)
5. LLM annotation — mock (free) or real Anthropic API (~$5–8)
6. SFT JSONL formatting with correct image paths

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ══════════════════════════════════════════════════════════
FORCE_RERUN      = True   # True: always rerun each step (overwrite outputs)
                           # False: skip a step if its output already exists
USE_MOCK_LLM     = True   # True: free mock annotations (no API key)
                           # False: real Claude API (~$5-8, needs ANTHROPIC_API_KEY)
NUSCENES_VERSION = "v1.0-mini"
MIN_RARITY_SCORE = 1
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install data-pipeline dependencies
# ⚠️  NEVER install numpy or pandas — Colab's pre-installed versions are compiled
#     against numpy 2.x ABI. Reinstalling them breaks pandas, opencv, jax, etc.
# nuscenes-devkit pins numpy<2; install with --no-deps then add real deps manually.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1

sys.path.insert(0, f"{REPO_ROOT}/src")

import numpy as np, pandas as pd, drivesense
print(f"✓ numpy {np.__version__} | pandas {pd.__version__} | DriveSense loaded")
from nuscenes.nuscenes import NuScenes
print("✓ nuscenes-devkit importable")

## 1. nuScenes Mini Download

nuScenes mini is ~1 GB. The download cell is skipped if already extracted (saves bandwidth).
Re-extraction is NOT triggered by `FORCE_RERUN` — delete the directory manually if needed.

In [ ]:
import os

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
MINI_DIR     = f"{NUSCENES_DIR}/v1.0-mini"
SAMPLES_DIR  = f"{NUSCENES_DIR}/samples"

if os.path.exists(MINI_DIR) and os.path.exists(SAMPLES_DIR):
    print("✓ nuScenes mini already extracted — skipping download.")
else:
    os.makedirs(NUSCENES_DIR, exist_ok=True)
    print("Downloading nuScenes mini (~1 GB)...")
    !wget -q "https://www.nuscenes.org/data/v1.0-mini.tgz" \
          -O /tmp/nuscenes_mini.tgz || true

    tgz = "/tmp/nuscenes_mini.tgz"
    if os.path.exists(tgz) and os.path.getsize(tgz) > 1_000_000:
        print("Extracting...")
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        !rm {tgz}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print("⚠ Auto-download failed. Upload v1.0-mini.tgz to Drive and run the fallback cell.")

In [ ]:
# FALLBACK: Extract from manually uploaded archive on Drive
import os, glob

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
already_done = (os.path.exists(f"{NUSCENES_DIR}/v1.0-mini") and
                os.path.exists(f"{NUSCENES_DIR}/samples"))

if already_done:
    print("✓ nuScenes already extracted — skipping.")
else:
    search_paths = [PROJECT_ROOT, "/content/drive/MyDrive"]
    candidates = []
    for base in search_paths:
        candidates += glob.glob(f"{base}/**/*.tgz", recursive=True)
    tgz = next((c for c in candidates
                if any(k in c.lower() for k in ("nuscenes", "mini", "v1.0"))), None)

    if tgz:
        print(f"Found: {tgz}")
        os.makedirs(NUSCENES_DIR, exist_ok=True)
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print(f"⚠ No archive found. Upload v1.0-mini.tgz to {PROJECT_ROOT}/ then rerun.")

In [ ]:
import os
from nuscenes.nuscenes import NuScenes

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
print(f"Contents of {NUSCENES_DIR}:")
!ls {NUSCENES_DIR} 2>/dev/null || echo "  (directory empty or missing)"

if os.path.exists(f"{NUSCENES_DIR}/v1.0-mini"):
    try:
        nusc = NuScenes(version="v1.0-mini", dataroot=NUSCENES_DIR, verbose=False)
        print(f"\n✓ nuScenes v1.0-mini loaded:")
        print(f"  Scenes      : {len(nusc.scene)}")
        print(f"  Samples     : {len(nusc.sample)}")
        print(f"  Annotations : {len(nusc.sample_annotation)}")
    except Exception as e:
        print(f"\n✗ Load failed: {e}")
else:
    print("\n⚠ v1.0-mini/ not found — complete the download step first.")

## 2. Rarity Filtering

Scores frames on 6 binary signals. `FORCE_RERUN=True` clears and reruns.
Output: `metadata.json` (array) → immediately converted to `metadata.jsonl` (lines).

In [ ]:
import os, json, shutil

FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
json_path     = f"{FILTER_OUTPUT}/metadata.json"
jsonl_path    = f"{FILTER_OUTPUT}/metadata.jsonl"

output_exists = os.path.exists(json_path)
if output_exists and not FORCE_RERUN:
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    print(f"✓ Already filtered ({n} frames). Set FORCE_RERUN=True to rebuild.")
else:
    if os.path.exists(FILTER_OUTPUT):
        shutil.rmtree(FILTER_OUTPUT)
    os.makedirs(FILTER_OUTPUT, exist_ok=True)
    print("Running nuScenes rarity filter...")
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version {NUSCENES_VERSION} \
        --output-dir {FILTER_OUTPUT} \
        --min-score {MIN_RARITY_SCORE} \
        2>&1

    # CRITICAL: convert JSON array → JSONL (unified builder expects JSONL)
    if os.path.exists(json_path) and not os.path.exists(jsonl_path):
        with open(json_path) as f:
            frames = json.load(f)
        with open(jsonl_path, "w") as f:
            for frame in frames:
                f.write(json.dumps(frame) + "\n")
        print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")

    if os.path.exists(jsonl_path):
        with open(jsonl_path) as f:
            n = sum(1 for _ in f)
        print(f"  ✓ metadata.jsonl: {n} frames")
    else:
        print("  ⚠ metadata.json not produced — check filter output above")

!ls -lh {FILTER_OUTPUT} 2>/dev/null

## 3. PySpark ETL

Distributed rarity scoring. `FORCE_RERUN=True` clears and reruns.

In [ ]:
import os, shutil

SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"

output_exists = os.path.exists(SPARK_OUTPUT) and bool(os.listdir(SPARK_OUTPUT)) if os.path.exists(SPARK_OUTPUT) else False
if output_exists and not FORCE_RERUN:
    print(f"✓ Spark output already exists. Set FORCE_RERUN=True to rebuild.")
    !ls {SPARK_OUTPUT}
else:
    if os.path.exists(SPARK_OUTPUT):
        shutil.rmtree(SPARK_OUTPUT)
    os.makedirs(SPARK_OUTPUT, exist_ok=True)
    os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-11-openjdk-amd64")
    print("Running PySpark rarity scoring pipeline...")
    !python scripts/run_spark_pipeline.py \
        --version {NUSCENES_VERSION} \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1
    print("✓ Spark pipeline complete.")
    !ls {SPARK_OUTPUT}

## 4. DADA-2000 (Optional)

53 GB accident dashcam dataset. Skip for mini/demo runs. `FORCE_RERUN=True` reruns extraction.

To include: upload `DADA-2000/` to `DriveSense-VLM/data/dada2000/DADA-2000/` on Drive.

In [ ]:
import os, shutil

DADA_ROOT   = f"{DATA_ROOT}/dada2000"
DADA_DATA   = f"{DADA_ROOT}/DADA-2000"
DADA_OUTPUT = f"{OUTPUTS_ROOT}/data/dada_extracted"

if os.path.exists(DADA_DATA) and os.listdir(DADA_DATA):
    print(f"✓ DADA-2000 found at {DADA_DATA}")
    output_exists = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    if output_exists and not FORCE_RERUN:
        print("✓ Already extracted. Set FORCE_RERUN=True to rebuild.")
    else:
        if os.path.exists(DADA_OUTPUT):
            shutil.rmtree(DADA_OUTPUT)
        os.makedirs(DADA_OUTPUT, exist_ok=True)
        print("Running DADA-2000 extraction...")
        !python scripts/run_dada_extraction.py \
            --dada-root {DADA_ROOT} \
            --output-dir {DADA_OUTPUT} \
            2>&1
        print("✓ DADA-2000 extraction complete.")
        !ls {DADA_OUTPUT}
else:
    print("⚠ DADA-2000 not found — skipping (mini run mode).")
    print(f"  To enable: upload data to {DADA_DATA}/")

## 5. Unified Dataset

Merges sources into stratified train/val/test splits, then combines into
`all_manifest.jsonl` for single-pass annotation. `FORCE_RERUN=True` rebuilds.

In [ ]:
import os, json, shutil

FILTER_OUTPUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
ALL_MANIFEST   = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"

output_exists = os.path.exists(f"{UNIFIED_OUTPUT}/train_manifest.jsonl")
if output_exists and not FORCE_RERUN:
    print("✓ Unified dataset already built. Set FORCE_RERUN=True to rebuild.")
    !ls {UNIFIED_OUTPUT}
else:
    if os.path.exists(UNIFIED_OUTPUT):
        shutil.rmtree(UNIFIED_OUTPUT)
    os.makedirs(UNIFIED_OUTPUT, exist_ok=True)

    HAS_DADA  = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    dada_flag = f"--dada-dir {DADA_OUTPUT}" if HAS_DADA else "--nuscenes-only"
    src_desc  = "nuScenes + DADA-2000" if HAS_DADA else "nuScenes only"
    print(f"Building unified dataset ({src_desc})...")
    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1

    print("\n✓ Split manifests:")
    for split in ("train", "val", "test"):
        p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
        if os.path.exists(p):
            with open(p) as f:
                n = sum(1 for _ in f)
            print(f"  {split}: {n} frames")
        else:
            print(f"  {split}: NOT FOUND")

# Always rebuild all_manifest.jsonl (cheap, needed by annotation pipeline)
print("\nBuilding all_manifest.jsonl...")
combined = []
for split in ("train", "val", "test"):
    p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                frame = json.loads(line)
                frame["split"] = split
                combined.append(frame)
with open(ALL_MANIFEST, "w") as f:
    for frame in combined:
        f.write(json.dumps(frame) + "\n")
print(f"  ✓ Combined {len(combined)} frames → all_manifest.jsonl")

## 6. LLM Annotation

`USE_MOCK_LLM=True` (default): free mock annotations, no API key needed.
`USE_MOCK_LLM=False`: real Claude API (~$5–8). Store key in Colab Secrets as `ANTHROPIC_API_KEY`.
`FORCE_RERUN=True`: clears previous annotations and reruns.

In [ ]:
import os, shutil

UNIFIED_OUTPUT    = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
ALL_MANIFEST      = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"

output_exists = os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json")
if output_exists and not FORCE_RERUN:
    print("✓ Annotation already done. Set FORCE_RERUN=True to rerun.")
else:
    if os.path.exists(ANNOTATION_OUTPUT):
        shutil.rmtree(ANNOTATION_OUTPUT)
    os.makedirs(ANNOTATION_OUTPUT, exist_ok=True)

    if USE_MOCK_LLM:
        print("Running annotation (mock mode — no API key needed)...")
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            --mock-llm \
            2>&1
    else:
        print("Running annotation (real Claude API)...")
        try:
            from google.colab import userdata
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
            print("✓ API key loaded from Colab Secrets")
        except Exception:
            print("⚠️ Could not load API key from Colab Secrets. Set manually:")
            print('   os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."')
            raise
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            2>&1

    if os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json"):
        print(f"\n✓ Annotation complete → {ANNOTATION_OUTPUT}/annotated_manifest.json")
    else:
        print("⚠ annotated_manifest.json not found — check output above")

## 7. Fix Image Paths & Create SFT Files

Always clears and rebuilds SFT files with correct absolute image paths.
Reads `metadata.json` to map `frame_id → image_path`.

In [ ]:
import json, os, shutil

ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
SFT_OUTPUT        = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_META       = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/metadata.json"
IMAGES_DIR        = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images"

# Always clear and rebuild SFT files
if os.path.exists(SFT_OUTPUT):
    shutil.rmtree(SFT_OUTPUT)
os.makedirs(SFT_OUTPUT, exist_ok=True)

# Build frame_id → image_path mapping from rarity-filter metadata
with open(FILTER_META) as f:
    filtered = json.load(f)
id_to_image = {}
for frame in filtered:
    fid = frame.get("sample_token", "")
    img = frame.get("exported_image_path", "")
    if img and not os.path.isabs(img):
        img = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/{img}"
    if not img or not os.path.exists(img):
        cam_path = frame.get("cam_front_path", "")
        if cam_path:
            candidate = f"{IMAGES_DIR}/{os.path.basename(cam_path)}"
            if os.path.exists(candidate):
                img = candidate
    id_to_image[fid] = img
mapped = sum(1 for v in id_to_image.values() if v)
print(f"Image mappings: {mapped}/{len(id_to_image)}")

# Load annotations
annot_path = f"{ANNOTATION_OUTPUT}/annotated_manifest.json"
assert os.path.exists(annot_path), f"No annotations at {annot_path}. Run annotation cell first."
with open(annot_path) as f:
    annotated = json.load(f)
print(f"Annotated frames: {len(annotated)}")

# Show annotation diversity
labels = set()
severities = set()
for frame in annotated:
    for h in frame.get("annotations", {}).get("hazards", []):
        labels.add(h.get("label", ""))
        severities.add(h.get("severity", ""))
print(f"Hazard labels  : {labels}")
print(f"Severities     : {severities}")

SYSTEM_PROMPT = (
    "You are DriveSense, an autonomous vehicle hazard detection system. "
    "Analyze the dashcam image and identify all safety-critical hazards. "
    "Output a structured JSON response with bounding boxes (normalized 0-1000), "
    "hazard classification, severity assessment, reasoning, and recommended action."
)
USER_PROMPT = (
    "Analyze this dashcam image for safety hazards. Identify all hazards with "
    "bounding boxes, classify each hazard, assess severity, explain your reasoning, "
    "and recommend an action. Respond with JSON only."
)

for split in ("train", "val", "test"):
    split_frames = [f for f in annotated if f.get("split") == split and f.get("annotations")]
    sft_path = f"{SFT_OUTPUT}/sft_{split}.jsonl"
    with open(sft_path, "w") as out:
        for frame in split_frames:
            fid      = frame.get("frame_id", "")
            img_path = id_to_image.get(fid, frame.get("image_path", ""))
            example  = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "image", "image": img_path},
                        {"type": "text",  "text": USER_PROMPT},
                    ]},
                    {"role": "assistant", "content": json.dumps(frame["annotations"])},
                ],
                "frame_id": fid,
                "source":   frame.get("source", ""),
                "split":    split,
            }
            out.write(json.dumps(example) + "\n")
    with open(sft_path) as f:
        count = sum(1 for _ in f)
    size = os.path.getsize(sft_path) / 1024
    print(f"  ✓ sft_{split}.jsonl: {count} examples ({size:.1f} KB)")

# Verify first example has a populated image path
with open(f"{SFT_OUTPUT}/sft_train.jsonl") as f:
    ex = json.loads(f.readline())
for msg in ex["messages"]:
    if isinstance(msg.get("content"), list):
        for item in msg["content"]:
            if item.get("type") == "image":
                p      = item["image"]
                exists = os.path.exists(p)
                print(f"\nImage path verification: {p}")
                print(f"Exists: {exists}")
                assert exists, f"Image not found! Check image path mapping."

print(f"\n✅ SFT data ready at {SFT_OUTPUT}")

## 8. Verification

Soft check of all pipeline outputs.

In [ ]:
import os, json

SFT_DIR     = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_OUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
UNIFIED_OUT = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATED   = f"{OUTPUTS_ROOT}/data/annotated"

checks = {
    "nuScenes filtered":    f"{FILTER_OUT}/metadata.json",
    "nuScenes JSONL":       f"{FILTER_OUT}/metadata.jsonl",
    "Spark processed":      f"{OUTPUTS_ROOT}/data/spark_processed",
    "Unified train":        f"{UNIFIED_OUT}/train_manifest.jsonl",
    "Unified combined":     f"{UNIFIED_OUT}/all_manifest.jsonl",
    "Annotated manifest":   f"{ANNOTATED}/annotated_manifest.json",
    "SFT train":            f"{SFT_DIR}/sft_train.jsonl",
    "SFT val":              f"{SFT_DIR}/sft_val.jsonl",
    "SFT test":             f"{SFT_DIR}/sft_test.jsonl",
}

print("=" * 56)
print("  Pipeline Verification")
print("=" * 56)
for label, path in checks.items():
    ok = os.path.exists(path)
    if ok and path.endswith(".jsonl"):
        with open(path) as f:
            n = sum(1 for _ in f)
        detail = f" ({n} records)"
    elif ok and path.endswith(".json"):
        detail = f" ({os.path.getsize(path)//1024} KB)"
    else:
        detail = ""
    print(f"  {'✓' if ok else '✗'}  {label}{detail}")
print("=" * 56)
print("\nReady for Notebook 01 (Training)")